In [34]:
import os
import pandas as pd
import numpy as np
import random
import warnings
warnings.filterwarnings(action = "ignore")
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import copy
import ab_sim as sim_mod
from scipy.special import gammaln
from scipy.special import gammaln
from scipy.optimize import minimize
from tqdm import tqdm
from scipy.stats import skewnorm
from skopt import gp_minimize
from skopt.space import Real

In [35]:
with open("initial_inputs_all.pkl", "rb") as input1:
    initial_inputs = pickle.load(input1)

initial_inputs

{'burn_in_days': 3650,
 'T_low': 0.01,
 'T_high': 0.15,
 'R_low': 0.04,
 'R_high': 0.08,
 'I_low': 1,
 'I_high': 6,
 'IC_low': 1,
 'IC_high': 10,
 'n_sims': 1000,
 'n_calls_gp': 1000,
 'n_random_starts': 500}

In [36]:
with open("case_inputs_all.pkl", "rb") as input2:
    farm = pickle.load(input2)
farm1 = farm['Farm1']
farm2 = farm['Farm2']
farm3 = farm['Farm3']
farm4 = farm['Farm4']

farm1.keys()

dict_keys(['internal_name', 'num_cows_to_add_burn_in', 'initial_infected', 'T', 'R', 'Incubation', 'target_curve', 'ndays_target', 'burn_in_days', 'no_calves', 'burn_in_farm'])

In [38]:
def dtw_distance(a, b):
    n, m = len(a), len(b)
    D = np.full((n+1, m+1), np.inf)
    D[0,0] = 0.0
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = abs(float(a[i-1]) - float(b[j-1]))
            D[i,j] = cost + min(D[i-1,j], D[i,j-1], D[i-1,j-1])
    return D[n,m]

# cross-correlation-based alignment: shift sim to align with obs, return minimal MAE across shifts
def crosscorr_alignment_distance(sim, obs, max_shift=None):
    # compute best shift of sim relative to obs to minimize MAE
    n = len(obs)
    if max_shift is None:
        max_shift = n // 2
    best_mae = np.mean(np.abs(sim - obs))  # zero shift baseline
    best_shift = 0
    for shift in range(-max_shift, max_shift+1):
        if shift == 0:
            continue
        # shift positive => sim delayed -> compare sim[shift:] with obs[:-shift]
        if shift > 0:
            s = sim[shift:]
            o = obs[:len(s)]
        else:
            s = sim[:len(sim)+shift]
            o = obs[-shift:len(obs)]
        if len(s) == 0:
            continue
        mae = np.mean(np.abs(s - o))
        if mae < best_mae:
            best_mae = mae
            best_shift = shift
    return best_mae, best_shift

# additional summary features for ML (for classifier)
def extract_features(curve):
    # raw stats + normalized peak/day + cumulative etc.
    curve = np.asarray(curve, dtype=float)
    total = curve.sum()
    peak = curve.max()
    peak_day = np.argmax(curve) + 1
    mean = curve.mean()
    std = curve.std()
    skew = 0.0
    if std > 0:
        skew = ((np.mean((curve-mean)**3)) / (std**3)) if std>0 else 0.0
    # normalized features
    feat = [total, peak, peak_day, mean, std, skew]
    return np.array(feat, dtype=float)

def plf(a,b):
    if len(a) != len(b):
        raise ValueError("inputs must have the same length.")
    epsilon = 1e-9
    log_fact = gammaln(b + 1)
    terms = a - b * np.log(a + epsilon) + log_fact
    weights = b + 1 #time varying weights proportional to b + 1 (avoids zero)
    return np.sum(weights * terms)/np.sum(weights)

In [39]:
# Dictionaries to store results
burn_in_farms = {}
initial_cows_by_case = {}
cases = ['Farm1', 'Farm2', 'Farm3', 'Farm4']
case_inputs = farm

# Loop over cases
for case in cases:
    inputs = case_inputs[case]
    
    # Create Farm instance for this case
    burn_in_farm = sim_mod.Farm(
        merge_dwell_df=sim_mod.merge_dwell,
        initial_infected=0,
        transmission_rate=0,
        recovery_rate=0
    )
    
    # Add cows to Pen1 and Pen5
    for _ in range(inputs["num_cows_to_add_burn_in"]):
        burn_in_farm.add_cow("Pen1", lactation=0, state='Susceptible')
        burn_in_farm.add_cow("Pen5", lactation=0, state='Susceptible')
    
    # Run simulation
    burn_in_farm.run_simulation(inputs["burn_in_days"], 0, 1)
    
    # Get initial cows
    initial_cows = {cow.id: cow for cow in burn_in_farm.cows.values() if cow.alive}
    
    # Store results
    burn_in_farms[case] = burn_in_farm
    initial_cows_by_case[case] = initial_cows
    
    # Print result
    print(f"Case {case}: Burn-in finished with {len(initial_cows)} alive cows.")

Case Farm1: Burn-in finished with 877 alive cows.
Case Farm2: Burn-in finished with 641 alive cows.
Case Farm3: Burn-in finished with 2619 alive cows.
Case Farm4: Burn-in finished with 2624 alive cows.


In [40]:
from functools import partial
# Define the objective function to minimize (using combined distance)
def objective_combined_distance(params, burn_in_farm_case, ndays_target_case, burn_in_days, target_curve_case, no_calves):
    T, R, incubation_float, seed_float = params
    incubation = int(round(incubation_float)) # gp_minimize works with floats, convert to int
    seed = int(round(seed_float)) # gp_minimize works with floats, convert to int

    # Ensure seed is at least 1
    seed = max(1, seed)

    sim_curve = sim_mod.simulate_counts(
        burn_in_farm_case,
        ndays_target_case,
        burn_in_days,
        seed,
        float(T),
        float(R),
        incubation,
        pens_to_mask=no_calves
    )

    # Calculate combined distance
    d_dtw = dtw_distance(sim_curve, target_curve_case)
    d_mae = np.mean(np.abs(sim_curve - target_curve_case))
    d_cc, _ = crosscorr_alignment_distance(sim_curve, target_curve_case, max_shift=ndays_target_case//2)
    distance = d_dtw + 5.0 * d_mae + 2.0 * d_cc
    return distance

def objective_combined_plf(params, burn_in_farm_case, ndays_target_case, burn_in_days, target_curve_case, no_calves):
    T, R, incubation_float, seed_float = params
    incubation = int(round(incubation_float)) # gp_minimize works with floats, convert to int
    seed = int(round(seed_float)) # gp_minimize works with floats, convert to int

    # Ensure seed is at least 1
    seed = max(1, seed)

    sim_curve = sim_mod.simulate_counts(
        burn_in_farm_case,
        ndays_target_case,
        burn_in_days,
        seed,
        float(T),
        float(R),
        incubation,
        pens_to_mask=no_calves
    )

    # Calculate combined distance
    distance = plf(sim_curve, target_curve_case)
    
    return distance

# Define the search space
space = [
    Real(initial_inputs['T_low'], initial_inputs['T_high'], name='T'),
    Real(initial_inputs['R_low'], initial_inputs['R_high'], name='R'),
    Real(initial_inputs['I_low'], initial_inputs['I_high'], name='I'), 
    Real(initial_inputs['IC_low'], initial_inputs['IC_high'], name='IC') 
]

In [23]:
optim_DM_case = {}

n_runs = 100  # number of independent optimizations per case

for case in cases:
    print(f"\nRunning gp_minimize for case {case}...")
    # dict_keys(['internal_name', 'num_cows_to_add_burn_in', 
    # 'initial_infected', 'T', 'R', 'Incubation', 'target_curve', 
    # 'ndays_target', 'burn_in_days', 'no_calves', 'burn_in_farm'])

    objective_partial = partial(
        objective_combined_distance,
        burn_in_farm_case=case_inputs[case]['burn_in_farm'],
        ndays_target_case=case_inputs[case]['ndays_target'],
        burn_in_days=case_inputs[case]['burn_in_days'],
        target_curve_case=case_inputs[case]['target_curve'],
        no_calves=case_inputs[case]['no_calves']
    )

    best_params_runs = []
    best_scores_runs = []

    # tqdm over the 100 optimization runs for this case
    for run in tqdm(range(n_runs), desc=f"Case {case} runs", leave=True):
        res_gp = gp_minimize(
            objective_partial,
            space,
            n_calls=100,
            n_random_starts=10,
            random_state=run,
            verbose=False
        )

        best_T, best_R, best_I_float, best_IC_float = res_gp.x
        best_I = int(round(best_I_float))
        best_IC = int(round(best_IC_float))

        best_params_runs.append({
            "T": best_T,
            "R": best_R,
            "I": best_I,
            "IC": best_IC
        })
        best_scores_runs.append(res_gp.fun)
        # print(f"Run {run+1:3d} best:")

    optim_DM_case[case] = {
        "best_params_runs": best_params_runs,
        "best_scores_runs": best_scores_runs
    }



Running gp_minimize for case Farm1...


Case Farm1 runs: 100%|██████████████████████| 100/100 [1:42:11<00:00, 61.32s/it]



Running gp_minimize for case Farm2...


Case Farm2 runs: 100%|██████████████████████| 100/100 [1:17:07<00:00, 46.27s/it]



Running gp_minimize for case Farm3...


Case Farm3 runs: 100%|████████████████████| 100/100 [19:29:18<00:00, 701.59s/it]



Running gp_minimize for case Farm4...


Case Farm4 runs: 100%|██████████████████████| 100/100 [2:11:03<00:00, 78.63s/it]


In [27]:
rows = []

for case, results in optim_DM_case.items():
    best_params_runs = results["best_params_runs"]
    best_scores_runs = results["best_scores_runs"]

    for run_idx, (params, score) in enumerate(zip(best_params_runs, best_scores_runs)):
        row = {
            "case": case,
            "run": run_idx,
            "T": params["T"],
            "R": params["R"],
            "I": params["I"],
            "IC": params["IC"],
            "score": score
        }
        rows.append(row)

df_optim_dm = pd.DataFrame(rows)
df_optim_dm.head()
df_optim_dm.to_csv("optimized_dm_100_iters.csv")

In [42]:
optim_PLF_case = {}

n_runs = 100  # number of independent optimizations per case

for case in cases:
    print(f"\nRunning gp_minimize for case {case}...")
    # dict_keys(['internal_name', 'num_cows_to_add_burn_in', 
    # 'initial_infected', 'T', 'R', 'Incubation', 'target_curve', 
    # 'ndays_target', 'burn_in_days', 'no_calves', 'burn_in_farm'])

    objective_partial = partial(
        objective_combined_plf,
        burn_in_farm_case=case_inputs[case]['burn_in_farm'],
        ndays_target_case=case_inputs[case]['ndays_target'],
        burn_in_days=case_inputs[case]['burn_in_days'],
        target_curve_case=case_inputs[case]['target_curve'],
        no_calves=case_inputs[case]['no_calves']
    )

    best_params_runs = []
    best_scores_runs = []

    # tqPLF over the 100 optimization runs for this case
    for run in tqdm(range(n_runs), desc=f"Case {case} runs", leave=True):
        res_gp = gp_minimize(
            objective_partial,
            space,
            n_calls=100,
            n_random_starts=10,
            random_state=run,
            verbose=False
        )

        best_T, best_R, best_I_float, best_IC_float = res_gp.x
        best_I = int(round(best_I_float))
        best_IC = int(round(best_IC_float))

        best_params_runs.append({
            "T": best_T,
            "R": best_R,
            "I": best_I,
            "IC": best_IC
        })
        best_scores_runs.append(res_gp.fun)
        # print(f"Run {run+1:3d} best:")

    optim_PLF_case[case] = {
        "best_params_runs": best_params_runs,
        "best_scores_runs": best_scores_runs
    }



Running gp_minimize for case Farm1...


Case Farm1 runs: 100%|██████████████████████| 100/100 [1:33:56<00:00, 56.36s/it]



Running gp_minimize for case Farm2...


Case Farm2 runs: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [1:18:59<00:00, 47.39s/it]



Running gp_minimize for case Farm3...


Case Farm3 runs: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [1:46:05<00:00, 63.66s/it]



Running gp_minimize for case Farm4...


Case Farm4 runs: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [1:28:16<00:00, 52.97s/it]


In [44]:
optim_PLF_case

{'Farm1': {'best_params_runs': [{'T': 0.025909372435158465,
    'R': 0.04722223724530129,
    'I': 2,
    'IC': 1},
   {'T': 0.12826440379262752, 'R': 0.08, 'I': 6, 'IC': 3},
   {'T': 0.07668534626663588, 'R': 0.06215210095246032, 'I': 6, 'IC': 1},
   {'T': 0.1275348915750223, 'R': 0.04979926267588569, 'I': 4, 'IC': 1},
   {'T': 0.059165924566420336, 'R': 0.04303731284953682, 'I': 3, 'IC': 1},
   {'T': 0.022323802354078066, 'R': 0.06842044764679948, 'I': 6, 'IC': 7},
   {'T': 0.01, 'R': 0.08, 'I': 4, 'IC': 5},
   {'T': 0.05955335192542806, 'R': 0.08, 'I': 6, 'IC': 4},
   {'T': 0.08847185908934935, 'R': 0.04, 'I': 5, 'IC': 5},
   {'T': 0.13889448326938755, 'R': 0.04894442665122764, 'I': 6, 'IC': 2},
   {'T': 0.03468845129334492, 'R': 0.05935129671439346, 'I': 3, 'IC': 2},
   {'T': 0.12228606675221126, 'R': 0.04, 'I': 5, 'IC': 4},
   {'T': 0.0841837694811774, 'R': 0.07414656828475746, 'I': 6, 'IC': 3},
   {'T': 0.02506188454331209, 'R': 0.07728043587677053, 'I': 6, 'IC': 6},
   {'T': 0.1

In [46]:
rows = []

for case, results in optim_PLF_case.items():
    best_params_runs = results["best_params_runs"]
    best_scores_runs = results["best_scores_runs"]

    for run_idx, (params, score) in enumerate(zip(best_params_runs, best_scores_runs)):
        row = {
            "case": case,
            "run": run_idx,
            "T": params["T"],
            "R": params["R"],
            "I": params["I"],
            "IC": params["IC"],
            "score": score
        }
        rows.append(row)

df_optim_plf = pd.DataFrame(rows)
df_optim_plf.head()
df_optim_plf.to_csv("optimized_plf_100_iters.csv")

In [49]:
df_optim_plf

,case,run,T,R,I,IC,score
0,Farm1,0,0.025909,0.047222,2,1,13.838026
1,Farm1,1,0.128264,0.080000,6,3,14.994716
2,Farm1,2,0.076685,0.062152,6,1,11.892281
3,Farm1,3,0.127535,0.049799,4,1,13.023392
4,Farm1,4,0.059166,0.043037,3,1,12.334867
...,...,...,...,...,...,...,...
395,Farm4,95,0.010690,0.040000,3,6,4.399772
396,Farm4,96,0.011688,0.046319,3,5,6.307697
397,Farm4,97,0.010000,0.040000,2,1,5.414214
398,Farm4,98,0.010000,0.045949,3,8,6.983447


In [48]:
import pickle

with open("optimized_plf_100_iters.pkl", 'wb') as file:
    pickle.dump(optim_PLF_case, file)
    
with open("optimized_dm_100_iters.pkl", 'wb') as file:
    pickle.dump(optim_DM_case, file)
    
    